In [1]:
from WindPy import w
import numpy as np
import pandas as pd
import datetime
import os
import json
from tqdm import tqdm

w.start()
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2024 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [2]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)

# Load Local Database

In [3]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

69

In [4]:
print(INDEX_DATA['000300.SH'].index[-1])

2025-04-01


# Update Existing Tickers

In [6]:
today = "2025-04-09"

for ticker in tqdm(INDEX_DATA):
    data = INDEX_DATA[ticker]
    start = (data.index[-1] + pd.Timedelta(days = 1)).strftime("%Y-%m-%d")
    
    new_data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]
    
    if new_data.columns[0] == 'OUTMESSAGE':
        print(ticker)
        continue
    
    INDEX_DATA[ticker] = pd.concat([data, new_data])
    INDEX_DATA[ticker].dropna(inplace = True)

100%|██████████████████████████████████████████████████████████████████████████████████| 69/69 [00:37<00:00,  1.84it/s]


In [7]:
INDEX_DATA['000300.SH'].tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-04-02,12.4376,3.4162,3884.3858
2025-04-03,12.4221,3.4267,3861.5034
2025-04-07,11.6708,3.6656,3589.4406
2025-04-08,11.9030,3.5966,3650.7590
2025-04-09,11.9692,3.5698,3686.7943


# Download New Tickers

In [8]:
# for ticker, start in tqdm(INDEX_START.items()):
#     if ticker in INDEX_DATA:
#         continue
        
#     assert ticker not in INDEX_DATA
    
#     data = w.wsd(ticker, 'pe_ttm,dividendyield2', start, today, '', usedf = True)[1]
    
#     junk = data['PE_TTM']
    
#     data.dropna(inplace = True)
    
#     INDEX_DATA[ticker] = data.copy(deep = True)

In [9]:
for ticker, data in INDEX_DATA.items():
    path = os.path.join("Data", f"{ticker}.csv")
    data.to_csv(path, index = True, encoding = "utf-8")